**NOTES**
- do not use classes - redundant
- save birds as points and directions 
- use magnitude to show birds of prey?
- kd tree for nearest neighbours

In [69]:
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import imageio
from matplotlib.colors import Normalize

In [70]:
N_prey = 5000 # number of prey birds
N_pred = 3 # number of predators
L = 32.0 # siez of the simulation box
r = 1.0 # interaction radius
r_b = 4.0 # predator scare distance
v0 = 2.0 # velocity
a = 0.15 # noisie amplitude
dt = 0.01 # time step

In [71]:
def plot_sim(positions, velocities):
    plt.quiver(
        positions[:,0], positions[:,1],
        velocities[:,0], velocities[:, 1]
    )
    plt.xlim(0,L)
    plt.ylim(0,L)
    plt.show()

In [ ]:
# initial conditions
positions_prey = np.random.uniform(0, L, size=(N_prey,2))
theta_prey = np.random.uniform(-np.pi, np.pi, size=N_prey)

positions_pred = np.random.uniform(0, L, size=(N_pred,2))
theta_pred = np.random.uniform(-np.pi, np.pi, size=N_pred)

# setup figure
fig, ax = plt.subplots(figsize=(9, 9))
frames = []

# simulation
for step in range(5000):
    
    # PREY INTERACTION
    prey_tree = cKDTree(positions_prey, boxsize=[L, L])
    dist = prey_tree.sparse_distance_matrix(prey_tree, r, output_type='coo_matrix')

    row, col = dist.row, dist.col

    complex_theta_prey = np.exp(1j * theta_prey)

    neighbor_sum = np.zeros(N_prey, dtype=complex)
    np.add.at(neighbor_sum, row, complex_theta_prey[col])
    neighbor_sum += complex_theta_prey

    theta_prey_new = np.angle(neighbor_sum)

    # PREDATOR INTERACTION
    tree_pred = cKDTree(positions_pred, boxsize=[L,L])
    _, idx_pred_to_prey = prey_tree.query(positions_pred, k=1)
    prey_target_pos = positions_prey[idx_pred_to_prey]

    # Minimum Image Convention - PBC
    vec_pred = prey_target_pos - positions_pred
    vec_pred = (vec_pred + L / 2) % L - L / 2
    
    theta_pred_target = np.arctan2(vec_pred[:, 1], vec_pred[:, 0])
    theta_pred_new = theta_pred_target

    # PREY ESCAPE INSTINCT
    # Let the KDTree calculate the wrapped distance for us
    dist_escape, idx_prey_to_pred = tree_pred.query(positions_prey, k=1)
    nearest_pred_pos = positions_pred[idx_prey_to_pred]

    # Minimum Image Convention - PBC
    vec_escape = positions_prey - nearest_pred_pos
    vec_escape = (vec_escape + L / 2) % L - L / 2

    escape_dir = np.arctan2(vec_escape[:, 1], vec_escape[:, 0])

    escape_mask = dist_escape < r_b
    theta_prey_new[escape_mask] = escape_dir[escape_mask]

    # NOISE
    theta_prey = theta_prey_new + a * np.random.uniform(-np.pi, np.pi, size=N_prey)
    theta_pred = theta_pred_new + a * np.random.uniform(-np.pi, np.pi, size=N_pred)

    # UPDATE POSITIONS
    vel_prey = v0 * np.column_stack((np.cos(theta_prey), np.sin(theta_prey)))
    vel_pred = v0 * 1.0 * np.column_stack((np.cos(theta_pred), np.sin(theta_pred)))

    positions_prey = (positions_prey + vel_prey * dt) % L
    positions_pred = (positions_pred + vel_pred * dt) % L 

    # render frames
    if step % 20 == 0:
        ax.clear()
        color = (theta_prey + np.pi) / (2 * np.pi)
        norm = Normalize(vmin=0, vmax=1)
        ax.quiver(
            positions_prey[:,0], positions_prey[:,1],
            vel_prey[:,0], vel_prey[:,1],
            color,
            cmap='magma',
            scale_units='xy',
            scale=5,
            width=0.002
        )

        ax.quiver(
            positions_pred[:,0], positions_pred[:,1],
            vel_pred[:,0], vel_pred[:,1],
            color='black',
            scale_units='xy',
            scale=2,
            width=0.0035
        )
        ax.set_xlim(0,L)
        ax.set_ylim(0,L)
        
        # save frame
        fig.canvas.draw()
        image = np.asarray(fig.canvas.buffer_rgba())
        frames.append(image.copy())

imageio.mimsave('birds_of_prey.gif', frames, fps=24, loop=0)
plt.close()

plt.close()